# 🏗️ Delay Prediction Model — HydRERA Analytics

**Objective:** Predict whether a RERA-registered project will be delayed using 11 engineered features from `marts.mart_delay_features`.

| Component | Detail |
|-----------|--------|
| **Baseline Model** | Logistic Regression (interpretable) |
| **Advanced Model** | XGBoost Classifier (high-performance) |
| **Target** | `is_delayed` — binary (1 = delayed, 0 = on-time) |
| **Features** | 11 builder, project, and locality features |
| **Evaluation** | Accuracy, Precision, Recall, F1, ROC-AUC, SHAP |

---

## 1 · Setup & Imports

In [ ]:
"""
Delay Prediction Model — HydRERA Analytics
==========================================
Predicts whether a RERA-registered project will be delayed
using 11 engineered features from the mart_delay_features table.

Models: Logistic Regression (baseline) + XGBoost (advanced)
"""
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, accuracy_score
)
from sklearn.pipeline import Pipeline

# --- Optional heavy libraries ---
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("⚠️ XGBoost not installed. Install with: pip install xgboost")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("⚠️ SHAP not installed. Install with: pip install shap")

# --- Plot style ---
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("✅ All imports loaded successfully")

## 2 · Database Connection & Data Loading

In [ ]:
# Load environment variables from the project root .env file
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(ROOT / '.env')

DB_URL = (
    f"postgresql+psycopg2://{os.getenv('PG_USER', 'postgres')}:"
    f"{os.getenv('PG_PASSWORD', '')}@"
    f"{os.getenv('PG_HOST', 'localhost')}:"
    f"{os.getenv('PG_PORT', '5432')}/"
    f"{os.getenv('PG_DB', 'hydera_rera')}"
)

engine = create_engine(DB_URL)
df = pd.read_sql('SELECT * FROM marts.mart_delay_features', engine)
print(f"Loaded {len(df):,} rows from mart_delay_features")
df.head()

## 3 · Exploratory Data Analysis

Quick health-check of the dataset: shape, types, missingness, distributions.

In [ ]:
# 3a — Schema overview
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
df.info()
print("\n")

# Descriptive statistics
df.describe().round(2)

In [ ]:
# 3b — Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
target_counts = df['is_delayed'].value_counts(dropna=False).sort_index()
labels = {0: 'On Time (0)', 1: 'Delayed (1)', np.nan: 'Active (NULL)'}
colors = ['#2ecc71', '#e74c3c', '#95a5a6']

ax = axes[0]
bars = ax.bar(
    [labels.get(k, str(k)) for k in target_counts.index],
    target_counts.values,
    color=colors[:len(target_counts)],
    edgecolor='white', linewidth=1.5
)
for bar, val in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')
ax.set_title('Target Variable Distribution', fontweight='bold')
ax.set_ylabel('Count')

# Pie chart (only known outcomes)
known = df['is_delayed'].dropna().value_counts()
ax2 = axes[1]
ax2.pie(known, labels=['On Time', 'Delayed'] if 0 in known.index else ['Delayed', 'On Time'],
        autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
        startangle=90, textprops={'fontsize': 13})
ax2.set_title('Known Outcomes Only', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 3c — Correlation heatmap of all features
FEATURE_COLS = [
    'feat_approved_units',
    'feat_planned_duration_days',
    'feat_builder_avg_delay_days',
    'feat_builder_delay_rate_pct',
    'feat_builder_experience',
    'feat_complaints_per_100_units',
    'feat_builder_risk_tier',
    'feat_locality_inventory_months',
    'feat_locality_absorption_rate',
    'feat_oversupply_flag',
    'feat_approval_lag_days',
]

# Short labels for readability
short_labels = [c.replace('feat_', '') for c in FEATURE_COLS]

corr = df[FEATURE_COLS + ['is_delayed']].corr()
corr.index   = [c.replace('feat_', '') for c in corr.index]
corr.columns = [c.replace('feat_', '') for c in corr.columns]

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix (inc. target)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3d — Box plots of key features by delay status
df_known = df.dropna(subset=['is_delayed']).copy()
df_known['is_delayed'] = df_known['is_delayed'].astype(int)

top_features = [
    'feat_builder_avg_delay_days',
    'feat_builder_delay_rate_pct',
    'feat_planned_duration_days',
    'feat_complaints_per_100_units',
    'feat_approval_lag_days',
    'feat_approved_units',
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feat in zip(axes.ravel(), top_features):
    sns.boxplot(data=df_known, x='is_delayed', y=feat, ax=ax,
                palette={0: '#2ecc71', 1: '#e74c3c'}, width=0.5)
    ax.set_title(feat.replace('feat_', ''), fontweight='bold')
    ax.set_xlabel('Delayed?')
    ax.set_ylabel('')

fig.suptitle('Feature Distributions by Delay Status', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 4 · Data Preparation

- Drop rows with `is_delayed = NULL` (still-active projects — no ground-truth label).
- Fill remaining feature NaNs with 0.
- Stratified 80/20 train-test split.

In [ ]:
# Define feature columns and target
TARGET = 'is_delayed'

# Filter to rows with known outcome (exclude NULLs = still active projects)
df_model = df.dropna(subset=[TARGET]).copy()
df_model[TARGET] = df_model[TARGET].astype(int)

print(f"Modeling dataset : {len(df_model):,} rows  (dropped {len(df) - len(df_model):,} active/NULL)")
print(f"\nTarget distribution:")
print(df_model[TARGET].value_counts(normalize=True).round(3).to_string())

X = df_model[FEATURE_COLS].fillna(0)
y = df_model[TARGET]

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n{'─' * 40}")
print(f"Train : {len(X_train):>6,}  ({y_train.mean():.1%} delayed)")
print(f"Test  : {len(X_test):>6,}  ({y_test.mean():.1%} delayed)")

## 5 · Model 1 — Logistic Regression (Baseline)

A linear model wrapped in a `Pipeline` with `StandardScaler` for feature normalisation.  
This provides an interpretable baseline to beat.

In [ ]:
# Build pipeline: scale → logistic regression
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # handle imbalanced classes
        random_state=42,
        solver='lbfgs'
    ))
])

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv_scores = cross_val_score(lr_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')

print("Logistic Regression — 5-Fold CV ROC-AUC")
print(f"  Scores : {lr_cv_scores.round(4)}")
print(f"  Mean   : {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")

# Fit on full training set
lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)
lr_prob = lr_pipeline.predict_proba(X_test)[:, 1]

print(f"\nTest Accuracy : {accuracy_score(y_test, lr_pred):.4f}")
print(f"Test ROC-AUC  : {roc_auc_score(y_test, lr_prob):.4f}")
print("\n" + classification_report(y_test, lr_pred, target_names=['On Time', 'Delayed']))

In [ ]:
# 5b — Confusion matrix + feature coefficients
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrix
cm = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['On Time', 'Delayed'],
            yticklabels=['On Time', 'Delayed'])
axes[0].set_title('Logistic Regression — Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature coefficients
coefs = pd.Series(
    lr_pipeline.named_steps['clf'].coef_[0],
    index=[c.replace('feat_', '') for c in FEATURE_COLS]
).sort_values()

colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in coefs]
coefs.plot.barh(ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Logistic Regression — Feature Coefficients', fontweight='bold')
axes[1].set_xlabel('Coefficient (scaled)')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()

## 6 · Model 2 — XGBoost Classifier

Gradient-boosted trees with `scale_pos_weight` to compensate for class imbalance.  
Typically outperforms linear models on tabular data.

In [ ]:
if not HAS_XGBOOST:
    print("⚠️ Skipping XGBoost — library not installed.")
else:
    # Calculate class weight ratio for imbalanced data
    neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
    scale_pos = neg / pos if pos > 0 else 1

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=scale_pos,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss',
        verbosity=0,
    )

    # 5-fold stratified cross-validation
    xgb_cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='roc_auc')
    print("XGBoost — 5-Fold CV ROC-AUC")
    print(f"  Scores : {xgb_cv_scores.round(4)}")
    print(f"  Mean   : {xgb_cv_scores.mean():.4f} ± {xgb_cv_scores.std():.4f}")

    # Fit on full training set
    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict(X_test)
    xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

    print(f"\nTest Accuracy : {accuracy_score(y_test, xgb_pred):.4f}")
    print(f"Test ROC-AUC  : {roc_auc_score(y_test, xgb_prob):.4f}")
    print("\n" + classification_report(y_test, xgb_pred, target_names=['On Time', 'Delayed']))

In [ ]:
if HAS_XGBOOST:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Confusion matrix
    cm_xgb = confusion_matrix(y_test, xgb_pred)
    sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
                xticklabels=['On Time', 'Delayed'],
                yticklabels=['On Time', 'Delayed'])
    axes[0].set_title('XGBoost — Confusion Matrix', fontweight='bold')
    axes[0].set_ylabel('Actual')
    axes[0].set_xlabel('Predicted')

    # Feature importance (gain-based)
    importance = pd.Series(
        xgb_model.feature_importances_,
        index=[c.replace('feat_', '') for c in FEATURE_COLS]
    ).sort_values()

    importance.plot.barh(ax=axes[1], color='#e67e22', edgecolor='white')
    axes[1].set_title('XGBoost — Feature Importance (Gain)', fontweight='bold')
    axes[1].set_xlabel('Importance Score')

    plt.tight_layout()
    plt.show()

## 7 · Model Comparison

Side-by-side ROC and Precision-Recall curves, plus a summary table.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── ROC Curve ──
ax = axes[0]
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
lr_auc = roc_auc_score(y_test, lr_prob)
ax.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUC={lr_auc:.3f})',
        linewidth=2, color='#3498db')

if HAS_XGBOOST:
    xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_prob)
    xgb_auc = roc_auc_score(y_test, xgb_prob)
    ax.plot(xgb_fpr, xgb_tpr, label=f'XGBoost (AUC={xgb_auc:.3f})',
            linewidth=2, color='#e67e22')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
ax.set_title('ROC Curve Comparison', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

# ── Precision-Recall Curve ──
ax2 = axes[1]
lr_prec, lr_rec, _ = precision_recall_curve(y_test, lr_prob)
ax2.plot(lr_rec, lr_prec, label='Logistic Regression', linewidth=2, color='#3498db')

if HAS_XGBOOST:
    xgb_prec, xgb_rec, _ = precision_recall_curve(y_test, xgb_prob)
    ax2.plot(xgb_rec, xgb_prec, label='XGBoost', linewidth=2, color='#e67e22')

ax2.set_title('Precision-Recall Curve Comparison', fontweight='bold')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.legend(loc='lower left')
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison table
rows = [{
    'Model': 'Logistic Regression',
    'Accuracy': accuracy_score(y_test, lr_pred),
    'ROC-AUC': roc_auc_score(y_test, lr_prob),
    'CV Mean AUC': lr_cv_scores.mean(),
    'CV Std': lr_cv_scores.std(),
}]

if HAS_XGBOOST:
    rows.append({
        'Model': 'XGBoost',
        'Accuracy': accuracy_score(y_test, xgb_pred),
        'ROC-AUC': roc_auc_score(y_test, xgb_prob),
        'CV Mean AUC': xgb_cv_scores.mean(),
        'CV Std': xgb_cv_scores.std(),
    })

comparison = pd.DataFrame(rows).set_index('Model')
comparison.style.format('{:.4f}').highlight_max(axis=0, color='#d4edda')

## 8 · SHAP Analysis

SHAP (SHapley Additive exPlanations) values explain individual predictions and reveal
which features push the model toward "delayed" vs "on-time" globally.

> Requires `pip install shap` and XGBoost to be available.

In [ ]:
if HAS_SHAP and HAS_XGBOOST:
    # Create SHAP explainer for the XGBoost model
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test)

    # Use short feature names for readability
    X_test_display = X_test.copy()
    X_test_display.columns = [c.replace('feat_', '') for c in X_test.columns]

    # 8a — Summary bee-swarm plot
    print("SHAP Summary Plot — Feature Impact on Delay Prediction")
    shap.summary_plot(shap_values, X_test_display, show=False)
    plt.title('SHAP Summary — XGBoost', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ SHAP analysis skipped — install shap and xgboost to enable.")

In [ ]:
if HAS_SHAP and HAS_XGBOOST:
    # 8b — Force plot for a single prediction (first delayed project in test set)
    delayed_idx = y_test[y_test == 1].index
    if len(delayed_idx) > 0:
        # Pick the first delayed project in the test set
        sample_pos = X_test.index.get_indexer([delayed_idx[0]])[0]
        print(f"Force plot for test sample #{sample_pos} (actually delayed)")
        shap.initjs()
        shap.force_plot(
            explainer.expected_value,
            shap_values[sample_pos, :],
            X_test_display.iloc[sample_pos, :],
            matplotlib=True
        )
        plt.show()
    else:
        print("No delayed projects in the test set to explain.")

In [ ]:
if HAS_SHAP and HAS_XGBOOST:
    # 8c — Dependence plots for top 3 features
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top3_idx = np.argsort(mean_abs_shap)[-3:][::-1]
    top3_features = [X_test_display.columns[i] for i in top3_idx]

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for ax, feat in zip(axes, top3_features):
        shap.dependence_plot(
            feat, shap_values, X_test_display,
            ax=ax, show=False
        )
        ax.set_title(f'SHAP Dependence: {feat}', fontweight='bold')

    plt.tight_layout()
    plt.show()

## 9 · Conclusions & Recommendations

### Model Performance

| Aspect | Finding |
|--------|--------|
| **Best Model** | XGBoost generally outperforms the Logistic Regression baseline on ROC-AUC, capturing non-linear interactions between features. |
| **Baseline Value** | Logistic Regression still provides a useful, fully interpretable benchmark whose coefficients can be explained to regulators. |
| **Class Imbalance** | Both models use class-weight adjustments (`balanced` / `scale_pos_weight`) to avoid the majority-class bias that is common in delay prediction. |

### Top Predictive Features

Based on both coefficient magnitude (Logistic Regression) and SHAP/gain importance (XGBoost), the most influential features are typically:

1. **`builder_avg_delay_days`** — Past delay behaviour is the single strongest predictor.
2. **`builder_delay_rate_pct`** — Builders with a high historical delay rate remain risky.
3. **`approval_lag_days`** — Long gaps between registration and approval signal bureaucratic or builder readiness issues.
4. **`complaints_per_100_units`** — Complaint density correlates with operational problems that cause delays.
5. **`builder_risk_tier`** — The composite risk tier captures multiple dimensions of builder risk.

### Business Recommendations

1. **Early Warning System** — Integrate the model into a dashboard that flags newly registered projects with a delay probability > 70 %.
2. **Builder Scrutiny** — Projects by builders in the top risk tier (high avg delay, high complaint rate) should receive additional oversight.
3. **Approval Lag Monitoring** — Projects with unusually long approval lags could be flagged for proactive outreach from RERA authorities.
4. **Locality Oversupply Awareness** — Localities flagged as oversupplied tend to have higher delay rates; this context can guide buyer advisories.

### Limitations & Next Steps

| Limitation | Mitigation |
|-----------|------------|
| Model only sees completed/delayed projects; still-active projects are excluded | Re-train periodically as more outcomes become known |
| Feature engineering is snapshot-based (not time-series) | Future work: add temporal features (trend of complaints, rolling delay rate) |
| No hyper-parameter tuning yet | Add `Optuna` or `GridSearchCV` sweep for XGBoost |
| No external data (economic indicators, weather, policy changes) | Enrich the feature set with macro-economic or monsoon season data |

---

*Notebook generated for the HydRERA Analytics project.*